In [ ]:
import os, subprocess, sys

def ensure_java17():
    try:
        jh = subprocess.check_output(
            ["/usr/libexec/java_home", "-v", "17"]
        ).decode().strip()
        os.environ["JAVA_HOME"] = jh
        os.environ["PATH"] = f"{jh}/bin:" + os.environ["PATH"]
        print("JAVA_HOME =", os.environ["JAVA_HOME"])
    except Exception as e:
        print("No pude obtener JAVA_HOME automáticamente:", e)
    v = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print("\njava -version (stderr):\n", v.stderr or v.stdout)

ensure_java17()

JAVA_HOME = /opt/homebrew/Cellar/openjdk@17/17.0.16/libexec/openjdk.jdk/Contents/Home

java -version (stderr):
 openjdk version "17.0.16" 2025-07-15
OpenJDK Runtime Environment Homebrew (build 17.0.16+0)
OpenJDK 64-Bit Server VM Homebrew (build 17.0.16+0, mixed mode, sharing)



## Cargar el RDD y exploración básica

In [6]:
from pyspark.sql import SparkSession
from pathlib import Path
spark = (
    SparkSession.builder
    .appName("MiApp")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

working_dir = Path.cwd()
txt_path = working_dir / "constitution.txt"

rdd = sc.textFile(str(txt_path)) 

print("Tipo de rdd:", type(rdd))

print("\nMuestra de 3 líneas (RDD.take):")
for i, line in enumerate(rdd.take(3), start=1):
    print(f"{i:>2}: {line}")

num_lineas = rdd.count()
print(f"\nTotal de líneas en el documento: {num_lineas}")

Tipo de rdd: <class 'pyspark.rdd.RDD'>

Muestra de 3 líneas (RDD.take):
 1: We the People of the United States, in Order to form a more perfect 
 2: Union, establish Justice, insure domestic Tranquility, provide for the 
 3: common defence, promote the general Welfare, and secure the Blessings of 

Total de líneas en el documento: 649


## Word Count (fase A): map vs flatMap y conteo de palabras

In [7]:
# 1) Con map
splitted_lines = rdd.map(lambda line: line.split(' '))
print("Ejemplo con map (3 elementos): cada elemento es una LISTA de palabras:")
for i, lst in enumerate(splitted_lines.take(3), start=1):
    print(f"{i}: {lst}")

count_map_elements = splitted_lines.count()
print(f"\nElementos en splitted_lines (debería ≈ # de líneas): {count_map_elements}")

# 2) Con flatMap
words_rdd_raw = rdd.flatMap(lambda line: line.split(' '))
raw_count = words_rdd_raw.count()

# Mejora: strip() antes de split para reducir vacíos
words_rdd = rdd.flatMap(lambda line: line.strip().split(' '))
total_words_incl_empty = words_rdd.count()
empty_words = words_rdd.filter(lambda w: w == "").count()

# Quita explícitamente vacíos para un conteo real de palabras
words_rdd_clean = words_rdd.filter(lambda w: w != "")
clean_count = words_rdd_clean.count()

print(f"\nConteo con flatMap (raw, puede incluir vacíos): {raw_count}")
print(f"Conteo tras strip().split(' '): {total_words_incl_empty} (vacíos: {empty_words})")
print(f"Conteo LIMPIO (sin vacíos): {clean_count}")

print("\nPrimeras 10 palabras (limpias):", words_rdd_clean.take(10))

Ejemplo con map (3 elementos): cada elemento es una LISTA de palabras:
1: ['We', 'the', 'People', 'of', 'the', 'United', 'States,', 'in', 'Order', 'to', 'form', 'a', 'more', 'perfect', '']
2: ['Union,', 'establish', 'Justice,', 'insure', 'domestic', 'Tranquility,', 'provide', 'for', 'the', '']
3: ['common', 'defence,', 'promote', 'the', 'general', 'Welfare,', 'and', 'secure', 'the', 'Blessings', 'of', '']

Elementos en splitted_lines (debería ≈ # de líneas): 649

Conteo con flatMap (raw, puede incluir vacíos): 8435
Conteo tras strip().split(' '): 7763 (vacíos: 140)
Conteo LIMPIO (sin vacíos): 7623

Primeras 10 palabras (limpias): ['We', 'the', 'People', 'of', 'the', 'United', 'States,', 'in', 'Order', 'to']


## Limpiar a solo alfanumérico (.isalnum) y hallar la palabra más larga

In [8]:
# 1) Dejar solo caracteres alfanuméricos en cada token (usando .isalnum en cada char)
words_alpha_rdd = (
    words_rdd_clean
    .map(lambda w: ''.join(ch for ch in w if ch.isalnum()))
    .filter(lambda w: w != '')
)

print("Ejemplo (10 palabras limpias alfanuméricas):", words_alpha_rdd.take(10))
print("Total de palabras alfanuméricas:", words_alpha_rdd.count())

# 2) Palabra más larga con reduce
longest = words_alpha_rdd.reduce(lambda a, b: a if len(a) > len(b) else b)
print(f"Palabra más larga: {longest!r} (longitud: {len(longest)})")

# versión en minúsculas
words_alpha_lower = words_alpha_rdd.map(lambda w: w.lower())
print("Ejemplo lower (10):", words_alpha_lower.take(10))

Ejemplo (10 palabras limpias alfanuméricas): ['We', 'the', 'People', 'of', 'the', 'United', 'States', 'in', 'Order', 'to']
Total de palabras alfanuméricas: 7610
Palabra más larga: 'constitutionally' (longitud: 16)
Ejemplo lower (10): ['we', 'the', 'people', 'of', 'the', 'united', 'states', 'in', 'order', 'to']


## Key-Value RDD y Top-5 palabras más frecuentes (sin filtrar stopwords)

In [15]:
# 1) Par (key, value) = (palabra, 1)
keyval_rdd = words_alpha_lower.map(lambda w: (w, 1))

# 2) Conteo por palabra
wordcount = keyval_rdd.reduceByKey(lambda a, b: a + b)

# 3) Ordenar por frecuencia (desc)
sorted_counts = (
    wordcount
    .map(lambda kv: (kv[1], kv[0]))     # (count, word)
    .sortByKey(ascending=False)         # mayor a menor
)

top5 = sorted_counts.take(5)
print("Top-5 (incluye stopwords):")
for rank, (cnt, word) in enumerate(top5, start=1):
    print(f"{rank}. {word} -> {cnt}")

# cuántas palabras únicas hay:
distinct_words = wordcount.count()
print("\nPalabras únicas (distinct):", distinct_words)

Top-5 (incluye stopwords):
1. the -> 726
2. of -> 494
3. shall -> 306
4. and -> 264
5. to -> 202

Palabras únicas (distinct): 1170


## Filtrar stopwords y mostrar Top-5 limpio

In [17]:
# Lista base de stopwords 
stopwords = {
    # comunes
    'the','of','and','to','in','a','is','for','on','with','that','by','be','as','are','at','from','or','an','it',
    'this','which','we','you','your','our','their','they','he','she','his','her','them','was','were','been','being',
    'but','not','no','do','does','did','so','if','than','then','there','here','such','into','out','over','under',
    'up','down','may','might','can','could','would','should','shall','will',
    'i','me','my','mine','us','ours','yours','theirs',
    # contracciones típicas
    's','t','d','ll','re','ve','m',

    'state','states','united','article','section','amendment','congress'
}

# Filtrar el RDD de conteo por palabra
filtered_wordcount = wordcount.filter(lambda kv: (kv[0] not in stopwords) and (len(kv[0]) >= 2))

# Ordenar por frecuencia desc 
sorted_counts_nostop = (
    filtered_wordcount
    .map(lambda kv: (kv[1], kv[0]))     # (count, word)
    .sortByKey(ascending=False)
)

top5_nostop = sorted_counts_nostop.take(5)
print("Top-5 SIN stopwords:")
for rank, (cnt, word) in enumerate(top5_nostop, start=1):
    print(f"{rank}. {word} -> {cnt}")

distinct_no_stop = filtered_wordcount.count()
print("\nPalabras únicas tras filtro (sin stopwords):", distinct_no_stop)

top20_nostop = sorted_counts_nostop.take(20)
# print(top20_nostop)

Top-5 SIN stopwords:
1. president -> 110
2. any -> 79
3. have -> 63
4. all -> 41
5. law -> 39

Palabras únicas tras filtro (sin stopwords): 1098


## Exploración básica
- **¿Qué tipo de data almacena el RDD cargado con `sc.textFile`?**  
  Se obtiene un `RDD[str]` donde **cada elemento corresponde a una línea** del archivo de texto.
- **Muestra (`take(3)`)**: se muestran 3 líneas completas del documento.
- **¿Cuántas líneas hay en el documento?**  
  Con `rdd.count()` se contabilizan **649 líneas**.

## `map` vs `flatMap`
- Con `map(lambda line: line.split(' '))` se obtiene un `RDD[list[str]]`: **cada elemento es una lista** de palabras de una línea.  
  Por ello, `splitted_lines.count()` ≈ **número de líneas**, no número de palabras.
- Con `flatMap(lambda line: line.strip().split(' '))` se obtiene un `RDD[str]` **de palabras**.  
  Dado que pueden aparecer tokens vacíos por espacios, se filtran con `.filter(lambda w: w != "")`.

## Limpieza de tokens con `.isalnum()`
- El enunciado solicita “palabras formadas exclusivamente por caracteres alfanuméricos”.  
  Se implementa una limpieza que **elimina** caracteres no alfanuméricos de cada token (`ch.isalnum()`), y también se documenta la variante que **convierte no alfanuméricos a espacio** (para evitar unir palabras con guion).
- Efecto esperado: sin este cuidado, `Vice-President` podría contarse como `VicePresident`. Con la versión que transforma no alfanuméricos en espacio, se separa en `["Vice","President"]`.

## Acciones vs Transformaciones
- **Transformaciones (lazy)** utilizadas: `map`, `flatMap`, `filter`, `reduceByKey`, `sortByKey`. No ejecutan trabajo hasta invocar una acción.
- **Acciones**: `count`, `take`, `reduce`. Estas **disparan** la ejecución del plan.

## Palabra más larga
- Con `reduce(lambda a,b: a if len(a)>len(b) else b)` la palabra más larga resultó ser:  
  **`constitutionally`** (longitud **16**).

## Conteos de palabras
1) **Sin filtrar stopwords** (Key-Value RDD):  
   Pipeline: `(palabra → 1)` → `reduceByKey` → invertir a `(count, word)` → `sortByKey(desc)` → `take(5)`.  
   Se espera observar artículos y conjunciones frecuentes (p. ej., `the`, `of`, `and`, …).

2) **Excluyendo stopwords**:  
   Se aplica un conjunto de stopwords comunes en inglés (artículos, pronombres, auxiliares, contracciones).  
   **Nota**: el **Top-5** resultante depende (a) de la estrategia de tokenización/limpieza y (b) de la lista de stopwords elegida; por ello son normales pequeñas variaciones en el ranking.

## Observaciones finales
- Los **RDDs** son inmutables y con **evaluación perezosa**; el plan se materializa al invocar una **acción**.  
- En tareas de texto, la definición de “palabra” impacta el resultado (espacios, guiones, puntuación). Usar `.isalnum()` cumple el requerimiento y deja documentada la decisión de limpieza.